In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import trimesh
from scipy.spatial import Delaunay
from collections import deque

from mc_tables import edgeTable, triTable

from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_FACE
from OCC.Core.BRep import BRep_Tool
from OCC.Core.TopLoc import TopLoc_Location
from OCC.Core.TopoDS import topods

pio.renderers.default = "browser"


stp_path = r"C:\Users\ZenBook\Desktop\Arms.stp"
reader = STEPControl_Reader()
if reader.ReadFile(stp_path) != 1: raise Exception(" Failed to load STEP")
reader.TransferRoots()
shape = reader.OneShape()

BRepMesh_IncrementalMesh(shape, 0.5)
verts_stp, faces_stp, offset = [], [], 0
exp = TopExp_Explorer(shape, TopAbs_FACE)
while exp.More():
    tri = BRep_Tool.Triangulation(topods.Face(exp.Current()), TopLoc_Location())
    if tri:
        for i in range(1, tri.NbNodes() + 1):
            p = tri.Node(i); verts_stp.append([p.X(), p.Y(), p.Z()])
        for i in range(1, tri.NbTriangles() + 1):
            i1, i2, i3 = tri.Triangle(i).Get()
            faces_stp.append([offset + i1 - 1, offset + i2 - 1, offset + i3 - 1])
        offset += tri.NbNodes()
    exp.Next()

mesh = trimesh.Trimesh(vertices=np.array(verts_stp), faces=np.array(faces_stp))
mesh.vertices -= mesh.vertices.mean(axis=0)
mesh.vertices /= np.max(np.linalg.norm(mesh.vertices, axis=1))


pitch = 0.012 
vox = mesh.voxelized(pitch)
matrix = vox.matrix.astype(np.uint8)

m = np.pad(matrix, 1, mode="constant", constant_values=0)
outside = np.zeros_like(m); q = deque([(0, 0, 0)]); outside[0,0,0] = 1
while q:
    x0, y0, z0 = q.popleft()
    for dx, dy, dz in [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]:
        nx, ny, nz = x0+dx, y0+dy, z0+dz
        if 0 <= nx < m.shape[0] and 0 <= ny < m.shape[1] and 0 <= nz < m.shape[2]:
            if outside[nx,ny,nz] == 0 and m[nx,ny,nz] == 0:
                outside[nx,ny,nz] = 1; q.append((nx,ny,nz))

F = ((m == 1) | (outside == 0))[1:-1, 1:-1, 1:-1].astype(float)
nx_s, ny_s, nz_s = F.shape


filled_idx = np.argwhere(F == 1)
cube_x, cube_y, cube_z = [], [], []
for ix, iy, iz in filled_idx[::12]: # Խտությունը ըստ քո ոճի
    base = np.array([ix, iy, iz])
    corners = np.array([base+[0,0,0], base+[1,0,0], base+[1,1,0], base+[0,1,0],
                        base+[0,0,1], base+[1,0,1], base+[1,1,1], base+[0,1,1]])
    c_world = trimesh.transform_points(corners, vox.transform)
    edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
    for e in edges:
        p1, p2 = c_world[e[0]], c_world[e[1]]
        cube_x += [p1[0], p2[0], None]; cube_y += [p1[1], p2[1], None]; cube_z += [p1[2], p2[2], None]


cubeVerts = [(0,0,0),(1,0,0),(1,1,0),(0,1,0),(0,0,1),(1,0,1),(1,1,1),(0,1,1)]
edgeIndex = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]

def interpolate(p1, p2, v1, v2):
    if abs(v1-v2) < 1e-6: return p1
    t = abs(v1 - 0.5) / (abs(v1 - v2) + 1e-9)
    return p1 + t * (p2 - p1)

mc_verts, mc_faces = [], []
for i in range(nx_s - 1):
    for j in range(ny_s - 1):
        for k in range(nz_s - 1):
            vals = [F[i+dx, j+dy, k+dz] for dx, dy, dz in cubeVerts]
            cubeindex = 0
            for n in range(8):
                if vals[n] > 0.5: cubeindex |= 1 << n
            if edgeTable[cubeindex] == 0: continue
            vertlist = [None] * 12
            cube_pts = [np.array([i+dx, j+dy, k+dz]) for dx, dy, dz in cubeVerts]
            for e, (a, b) in enumerate(edgeIndex):
                if edgeTable[cubeindex] & (1 << e):
                    vertlist[e] = interpolate(cube_pts[a], cube_pts[b], vals[a], vals[b])
            t = 0
            while triTable[cubeindex][t] != -1:
                idx_start = len(mc_verts)
                mc_verts.extend([vertlist[triTable[cubeindex][t]], vertlist[triTable[cubeindex][t+1]], vertlist[triTable[cubeindex][t+2]]])
                mc_faces.append([idx_start, idx_start+1, idx_start+2])
                t += 3

mc_verts_world = trimesh.transform_points(np.array(mc_verts), vox.transform)
mc_faces = np.array(mc_faces)


indices = np.random.choice(len(mc_verts_world), 800, replace=False)
pts_dl = mc_verts_world[indices]
tri_dl = Delaunay(pts_dl)
dl_x, dl_y, dl_z = [], [], []
for s in tri_dl.simplices:
    for i, j in [(0,1), (1,2), (2,0)]:
        p1, p2 = pts_dl[s[i]], pts_dl[s[j]]
        if np.linalg.norm(p1 - p2) < pitch * 3.5:
            dl_x += [p1[0], p2[0], None]; dl_y += [p1[1], p2[1], None]; dl_z += [p1[2], p2[2], None]

fig = go.Figure()


fig.add_trace(go.Scatter3d(x=cube_x, y=cube_y, z=cube_z, mode='lines',
                         line=dict(color="rgba(180,180,180,0.18)", width=1), name="Voxels"))


z_norm = (mc_verts_world[:, 2] - mc_verts_world[:, 2].min()) / (np.ptp(mc_verts_world[:, 2]) + 1e-9)
fig.add_trace(go.Mesh3d(x=mc_verts_world[:,0], y=mc_verts_world[:,1], z=mc_verts_world[:,2],
                       i=mc_faces[:,0], j=mc_faces[:,1], k=mc_faces[:,2],
                       intensity=z_norm, colorscale=[[0, "#fffdf8"], [0.5, "#d9c399"], [1, "#b69c73"]],
                       opacity=1.0, name="Manual MC Surface", lighting=dict(ambient=0.4, diffuse=0.8)))


fig.add_trace(go.Scatter3d(x=dl_x, y=dl_y, z=dl_z, mode='lines', line=dict(color="red", width=1.5), name="Delaunay"))
fig.add_trace(go.Scatter3d(x=pts_dl[:,0], y=pts_dl[:,1], z=pts_dl[:,2], mode='markers', marker=dict(size=2, color="blue"), name="Nodes"))

fig.update_layout(width=1200, height=900, scene=dict(aspectmode="data", bgcolor="white"), paper_bgcolor="white")
fig.show()
print(f" Գագաթների (կետերի) քանակը = {len(mc_verts_world)}")
print(f" Եռանկյունների քանակը = {len(mc_faces)}")

 Գագաթների (կետերի) քանակը = 47874
 Եռանկյունների քանակը = 15958
